# 12b — Authorization, caches, and isolation — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/12.md) · [Course map](../PLAN.md) · [Project](../../projects/stage12/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Test direct-object access
- Scope cache and state keys
- Prevent identity spoofing through content


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

Authentication answers who is calling; authorization checks whether that principal can perform this operation on this resource. Resource IDs are references, not permissions. A caller who guesses another tenant’s document ID must still be denied. Check authorization at every data/tool boundary, including background jobs and citations.

Caches can accidentally cross trust boundaries. A response key containing only question text can serve one user another user’s answer. Include tenant, effective permission version, index version, model/prompt version, and query. Invalidate when permissions change. A hash obscures a key but does not encrypt cached content.

Use deny-by-default behavior, private-by-default documents, and separate credentials for services. Credentials supplied in HTTP headers must be verified before constructing a principal. Do not accept a tenant identity from a model tool argument. Tests should include a permitted control case so a system that denies everything cannot pass isolation tests by accident.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
principal = {"user_id": "u1", "tenant": "A", "role": "reader"}
resource = {"id": "doc1", "tenant": "A", "public": False}
print(principal["tenant"] == resource["tenant"])


## Exercise 1: Read authorization

Return whether a reader/admin principal may read a public or same-tenant resource. Unknown roles are denied.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def can_read(principal, resource):
    return principal.get("role") in {"reader", "admin"} and (resource.get("public", False) or principal.get("tenant") == resource["tenant"])


In [ ]:
assert can_read({"role": "reader", "tenant": "A"}, {"tenant": "A"})
assert not can_read({"role": "admin", "tenant": "B"}, {"tenant": "A"})
assert not can_read({"role": "unknown", "tenant": "A"}, {"tenant": "A"})
print("Exercise 1: checks passed")


**Why this works:** An implicit global administrator role can break isolation even when ordinary users are scoped.


## Exercise 2: Cache identity

Return a deterministic key from tenant, permission_version, index_version, model_version, query using canonical JSON and SHA256.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def cache_key(tenant, permission_version, index_version, model_version, query):
    import hashlib, json
    fields = [tenant, permission_version, index_version, model_version, query]
    return hashlib.sha256(json.dumps(fields, ensure_ascii=False).encode()).hexdigest()


In [ ]:
assert cache_key("A", 1, 1, "m", "q") != cache_key("B", 1, 1, "m", "q")
assert cache_key("A", 1, 1, "m", "q") != cache_key("A", 2, 1, "m", "q")
print("Exercise 2: checks passed")


**Why this works:** A cache can otherwise preserve revoked access or stale evidence.


## Exercise 3: Resource lookup

Return a resource by ID only when authorized; otherwise raise PermissionError, including missing IDs, to avoid distinguishing existence.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def get_document(resources, id, principal):
    resource = resources.get(id)
    if resource is None or not can_read(principal, resource): raise PermissionError("Not available")
    return resource


In [ ]:
resources = {"d": {"tenant": "A", "text": "private"}}
expect_error(PermissionError, lambda: get_document(resources, "d", {"role": "reader", "tenant": "B"}))
assert get_document(resources, "d", {"role": "reader", "tenant": "A"})["text"] == "private"
print("Exercise 3: checks passed")


**Why this works:** Consistent errors can reduce existence leaks; timing and other channels still require consideration.


## Independent transfer

Exercise every API endpoint as two users and two tenants. Test warm-cache access, revoked permissions, citation fetches, saved graph state, and backups. Produce a permission matrix with allowed and denied evidence.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Does an unguessable ID replace authorization?

   No. IDs can leak through logs, URLs, or shared links.

2. What belongs in cache identity?

   The effective access scope and every versioned input that can change the answer.


## Reading and review

- [Primary reference 1](https://genai.owasp.org/llm-top-10/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
